# Zoernikav — stain removal with Tri-Level Semi-Adaptive Thresholding

Implementation of

> N. Shobha Rani, B. J. Bipin Nair, M. Chandrajith, G. Hemantha Kumar & Jaume Fortuny, "Restoration of deteriorated text sections in ancient document images using a tri-level semi-adaptive thresholding technique", *Automatika* 63, no. 2 (2022): 378–398. DOI: [10.1080/00051144.2022.2042462](https://doi.org/10.1080/00051144.2022.2042462). Open access, CC BY 4.0.

applied to the stained folios of Adam Zoernikav's *Tractatus de processione Spiritus Sancti*.

**Pipeline (paper, Fig. 1)**

```
input ─► 3.1 pre-enhancement ─┬─► 3.2 gradient image Gd ─────────────► binary mask Bd ─┐
                              └─► greyscale Gs ─► 3.3 Level 1 ─► 3.4 Level 2 ─► 3.5 Level 3 ─► 3.6 pseudo-colouring ─► post-enhancement ─► binarised
```

**Notebook layout**

1. Setup
2. The paper's method, one section per subsection of §3
3. The full paper pipeline, with every intermediate image (as in Figs. 9–11)
4. Adaptation for Zoernikav's brown iron-gall ink (stain-masked Level 3 + a tone-preserving reading output)
5. Batch processing of every image in `StainImages/`
6. Baseline methods the paper compares against
7. The paper's evaluation metrics (§4), for when a ground-truth image is available

**Where the paper is ambiguous, the choice made here is stated in the cell.** There are four such places:

- eq. 1 (how T1 is averaged)
- eqs. 5–7 (whether pixels below each threshold are kept or set to 0)
- eqs. 7–8 (which partition Level 3 uses)
- the pseudo-colouring pseudocode (what happens outside the mask)

Each one is settled by the paper's own figures.

## 1. Setup

In [ ]:
import glob, os
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMAGE_DIR = 'StainImages'          # folder with the folio scans
OUT_DIR   = 'restored'             # results are written here
DEFAULT_IMAGE = os.path.join(IMAGE_DIR, 'Zoernikav folio212.jpg')

os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams['figure.dpi'] = 110

def to_rgb(img):
    """OpenCV loads BGR; matplotlib expects RGB."""
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img

def show(images, titles=None, cols=2, width=16):
    """Show images in a grid. Greyscale is drawn with a fixed 0-255 scale (no auto-contrast)."""
    n = len(images)
    rows = (n + cols - 1) // cols
    h, w = images[0].shape[:2]
    fig, axes = plt.subplots(rows, cols, figsize=(width, width / cols * h / w * rows + 0.4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, i in zip(axes, range(len(axes))):
        ax.axis('off')
        if i < n:
            img = images[i]
            if img.ndim == 2:
                ax.imshow(img, cmap='gray', vmin=0, vmax=255 if img.max() > 1 else 1, interpolation='lanczos')
            else:
                ax.imshow(to_rgb(img), interpolation='lanczos')
            if titles: ax.set_title(titles[i], fontsize=10)
    plt.tight_layout(); plt.show()

image_paths = sorted(p for ext in ('*.jpg', '*.jpeg', '*.png', '*.tif', '*.tiff')
                     for p in glob.glob(os.path.join(IMAGE_DIR, ext)))
path = DEFAULT_IMAGE if os.path.exists(DEFAULT_IMAGE) else image_paths[0]
I = cv2.imread(path)
print(f'{len(image_paths)} images found; working example: {path}  {I.shape}')
show([I], [os.path.basename(path)], cols=1, width=12)

## 2. The paper's method

### 3.1 Pre-enhancement

"Contrast stretching linear function that stretches the grey level of an image I to its full dynamic range", giving the enhanced image **I_E**.

This is implemented as MATLAB's `imadjust(I, stretchlim(I))`: each channel is stretched separately, and 1% of pixels at each end are saturated.

In [ ]:
def pre_enhance(bgr, low_pct=1.0, high_pct=99.0):
    out = np.empty_like(bgr)
    for c in range(3):
        ch = bgr[..., c].astype(np.float32)
        lo, hi = np.percentile(ch, [low_pct, high_pct])
        out[..., c] = np.clip((ch - lo) * 255.0 / max(hi - lo, 1e-6), 0, 255).astype(np.uint8)
    return out

IE = pre_enhance(I)
show([I, IE], ['I (original)', 'I_E (pre-enhanced)'])

### 3.2 Gradient image computation

Algorithm in §3.2:

1. RGB sliced image: $I_d = I_G - \frac{I_R + I_B}{2}$, computed on $I_E$
2. $I_{ref} = \text{Otsu}(I_d)$
3. Every pixel with $I_{ref}=1$ is set to 0 in all three channels of the **original** image $I$. The result is the gradient image $G_d$.

On yellow-buff paper, $G - (R+B)/2$ is highest on the clean paper, so Otsu marks the paper as 1 and $G_d$ keeps only ink and stain on a black background. This matches the paper's Figs. 4(d–f) and 10(b).

In [ ]:
def gradient_image(I, IE):
    b, g, r = cv2.split(IE.astype(np.float32))
    Id = g - (r + b) / 2.0
    Id_u8 = cv2.normalize(Id, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, Iref = cv2.threshold(Id_u8, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    Gd = I.copy()
    Gd[Iref == 1] = 0
    return Gd, Iref, Id_u8

Gd, Iref, Id = gradient_image(I, IE)
Gs = cv2.cvtColor(IE, cv2.COLOR_BGR2GRAY)          # greyscale workflow (Fig. 1)
Gd_gray = cv2.cvtColor(Gd, cv2.COLOR_BGR2GRAY)
print(f'I_ref marks {100 * Iref.mean():.0f}% of the page')
show([Id, Iref * 255, Gd, Gs], ['I_d = G - (R+B)/2', 'I_ref = Otsu(I_d)', 'G_d (gradient image)', 'G_s (greyscale of I_E)'])

### 3.3 Level 1 — uneven illumination, smears, foxing

$T_\mu^{(1)} = \frac{1}{n}\sum_{i \in \bar\vartheta} p(r_i)\,[G_d(:)]$ (eq. 1). It is computed over the grey levels of $G_d$ that actually occur: levels with $p(r_k) = 0$ are truncated. Level 0 is also excluded, because those are the pixels that $I_{ref}$ blanked.

Read literally, the $\frac{1}{n}$ makes $T_\mu^{(1)}$ the mean of the $n$ distinct grey levels present. That is the default (`mode='distinct'`), and on these folios it gives T1 ≈ 141–150, inside the 110–150 range the paper reports in Fig. 4. `mode='weighted'` gives the pixel-weighted mean instead.

Eq. 5 is $g_1 = 255$ if $G_s \ge T_\mu^{(1)}$. The prose says the transformation "separates the grey levels over a threshold to the maximum grey level **whilst keeping the grey levels below the threshold constant**", and Fig. 10(c) shows a grey stain surviving Level 1. So pixels below the threshold keep their grey value; they are not set to 0.

In [ ]:
def level1(Gs, Gd_gray, mode='distinct'):
    vals = Gd_gray[Gd_gray > 0]
    if vals.size == 0:
        vals = Gs.ravel()
    T1 = float(np.unique(vals).mean()) if mode == 'distinct' else float(vals.mean())
    g1 = Gs.copy()
    g1[Gs >= T1] = 255
    return g1, T1

g1, T1 = level1(Gs, Gd_gray)

hist = cv2.calcHist([Gd_gray], [0], None, [256], [0, 256]).ravel(); hist[0] = 0
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.bar(range(256), hist, width=1, color='0.45'); ax.axvline(T1, color='C3', lw=2)
ax.set_title(f'Histogram of G_d (level 0 removed) with T1 = {T1:.1f}  (cf. paper Fig. 4 g–j)')
ax.set_xlabel('grey level'); ax.set_ylabel('pixels'); plt.tight_layout(); plt.show()
show([Gs, g1], ['G_s', f'g1 — Level 1 (T1 = {T1:.1f})'])

### 3.4 Level 2 — impulse noise and moderate stains

The threshold $\tilde g_1$ is the median of the sorted distinct grey levels of $g_1$ (eq. 6).

As in Level 1, pixels at or above $\tilde g_1$ go to 255 and the rest keep their grey level. Eq. 6 writes "0 otherwise", but Level 3 slices the distinct grey levels of $g_2$, which would be meaningless if $g_2$ were already binary. Fig. 10(d) also shows grey tones in $g_2$.

In [ ]:
def level2(g1):
    T2 = float(np.median(np.unique(g1)))
    g2 = g1.copy()
    g2[g1 >= T2] = 255
    return g2, T2

g2, T2 = level2(g1)
show([g1, g2], [f'g1 — Level 1 (T1 = {T1:.1f})', f'g2 — Level 2 (T2 = {T2:.1f})'])

### 3.5 Level 3 — stain marks crossing text strokes (intensity slicing)

The sorted distinct grey levels of $g_2$ are split into four equal partitions $p_1 \dots p_4$. Pixels at or above the mean of one partition go to 255.

**The paper is inconsistent here.** The prose and eq. 7 say $\mu(p_2)$. The summation limits of eq. 8 ($l_{p_2+1} \dots l_{p_3}$) and Fig. 5 ("Mean(p3)") describe partition $p_3$. Both readings are available through `partition`. The paper's text (`'p2'`) is the default for the faithful run.

In [ ]:
def level3(g2, partition='p2'):
    levels = np.unique(g2)
    parts = np.array_split(levels, 4)
    idx = {'p2': 1, 'p3': 2}[partition]
    T3 = float(parts[idx].mean()) if parts[idx].size else float(levels.mean())
    g3 = g2.copy()
    g3[g2 >= T3] = 255
    return g3, T3

g3, T3 = level3(g2, 'p2')
g3_p3, T3_p3 = level3(g2, 'p3')
show([g2, g3, g3_p3], [f'g2 — Level 2', f'g3 — Level 3, mean(p2) = {T3:.1f}', f'g3 — Level 3, mean(p3) = {T3_p3:.1f}'], cols=3, width=18)

### 3.6 Pseudo-colouring and post-enhancement

Algorithm `pseudo_coloring(g3, Gd)`:

1. Temp = Otsu($G_d$).
2. Keep the connected components with area > S = 500 px. This gives the binary mask $B_d$ of degradation-obscured regions.
3. Compute $\Theta = \frac{1}{2}\,\text{mean}(g_3)$.
4. Inside $B_d$: $g_3 \ge \Theta \to 255$, otherwise 0. The result is $g_4$.

Outside $B_d$, the pseudocode's `else g3 = 0` cannot mean "paint everything black", since Figs. 9(g)–11(g) have white backgrounds. Here, outside the mask, whatever the three levels pushed to 255 is background and everything else is ink.

Post-enhancement is "morphological bridging operators followed by dilation". Bridging is MATLAB's `bwmorph(BW,'bridge')`: a 0-pixel becomes 1 if it has two 8-neighbours that are not connected to each other. Both operations are applied to the **ink**, as the foreground. Applied to the white background instead, they would erode the strokes.

**Diagnostic:** Θ ≈ 120 here, which is far above T3. Every pixel left after Level 3 is therefore already below Θ, and pseudo-colouring changes nothing on these folios. That is a property of the method on this ink, not a bug; the printout below reports it.

In [ ]:
def pseudo_colouring(g3, Gd_gray, min_area=500):
    _, temp = cv2.threshold(Gd_gray, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(temp, connectivity=8)
    keep = np.zeros(n, bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] > min_area
    Bd = keep[lab].astype(np.uint8)
    theta = 0.5 * float(g3.mean())
    g4 = np.where(g3 == 255, 255, 0).astype(np.uint8)
    inside = Bd == 1
    g4[inside] = np.where(g3[inside] >= theta, 255, 0)
    return g4, Bd, theta


def _bridge_lut():
    nb = [(-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1)]
    lut = np.zeros(256, np.uint8)
    for code_ in range(256):
        on = [i for i in range(8) if code_ >> i & 1]
        seen, comps = set(), 0
        for s in on:
            if s in seen: continue
            comps += 1; stack = [s]
            while stack:
                a = stack.pop()
                if a in seen: continue
                seen.add(a)
                stack += [b for b in on if b not in seen and
                          max(abs(nb[a][0] - nb[b][0]), abs(nb[a][1] - nb[b][1])) == 1]
        lut[code_] = comps >= 2
    return lut, nb

_LUT, _NB = _bridge_lut()

def bridge(mask):
    """MATLAB bwmorph(mask, 'bridge')."""
    m = np.pad(mask.astype(np.uint8), 1); h, w = mask.shape
    code_ = np.zeros((h, w), np.int32)
    for i, (dy, dx) in enumerate(_NB):
        code_ |= m[1 + dy:1 + dy + h, 1 + dx:1 + dx + w].astype(np.int32) << i
    return mask.astype(bool) | _LUT[code_].astype(bool)

def post_enhance(g4, dilate_size=2):
    ink = bridge(g4 == 0)
    if dilate_size > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_size, dilate_size))
        ink = cv2.dilate(ink.astype(np.uint8), k) > 0
    return np.where(ink, 0, 255).astype(np.uint8)


g4, Bd, theta = pseudo_colouring(g3, Gd_gray)
binary = post_enhance(g4)
changed = int(((g3 < 255) != (g4 == 0)).sum())
print(f'Theta = {theta:.1f}, T3 = {T3:.1f}; pixels changed by pseudo-colouring: {changed}')
show([Bd * 255, g4, binary], ['B_d — binary mask', 'g4 — pseudo-coloured', 'final (bridged + dilated)'], cols=3, width=18)

## 3. The full paper pipeline

`tsat()` runs §3.1–3.6 end to end. `levels=3` is the paper's method; `levels=1` or `levels=2` stop the cascade early. The figure reproduces the layout of the paper's Figs. 9–11.

In [ ]:
def tsat(bgr, levels=3, t1_mode='distinct', level3_partition='p2', min_area=500, dilate_size=2):
    IE = pre_enhance(bgr)
    Gd, Iref, Id = gradient_image(bgr, IE)
    Gs = cv2.cvtColor(IE, cv2.COLOR_BGR2GRAY)
    Gd_gray = cv2.cvtColor(Gd, cv2.COLOR_BGR2GRAY)
    g1, T1 = level1(Gs, Gd_gray, t1_mode)
    g2, T2 = level2(g1)
    g3, T3 = level3(g2, level3_partition)
    last = {1: g1, 2: g2, 3: g3}[levels]
    g4, Bd, theta = pseudo_colouring(last, Gd_gray, min_area)
    return dict(I=bgr, IE=IE, Id=Id, Iref=Iref, Gd=Gd, Gs=Gs, g1=g1, g2=g2, g3=g3,
                Bd=Bd, g4=g4, binary=post_enhance(g4, dilate_size),
                T=dict(T1=T1, T2=T2, T3=T3, theta=theta))

def show_stages(r):
    show([r['I'], r['Gd'], r['g1'], r['g2'], r['g3'], r['Bd'] * 255, r['binary']],
         ['(a) original', '(b) gradient image', '(c) Level 1', '(d) Level 2',
          '(e) Level 3', '(f) binary mask', '(g) binarised'], cols=4, width=20)

paper = tsat(I)
print({k: round(v, 1) for k, v in paper['T'].items()})
show_stages(paper)

## 4. Adaptation for Zoernikav's ink

The paper was tuned on DIBCO scans and palm leaves, where the ink is close to black. Zoernikav wrote in brown iron-gall ink, whose strokes sit at grey levels of up to about 70 after Level 2. That produces two effects on these folios:

- **Level 2 already removes the stain**, but strokes *inside* the stain come out heavy, because the stain darkens the paper around them.
- **Level 3 cleans those strokes but thins the text everywhere else.** Its threshold is a fixed fraction of the Level 2 range (about 3/8 for p2), which also cuts into the lighter half of the ink.

The paper's own remedy is the mask $B_d$, which restricts the stain-specific processing to stained regions. On these folios, though, Otsu($G_d$) marks text and stain together, not the stain alone. The adaptation below keeps the paper's three levels unchanged and only replaces the mask:

1. **Stain mask.** A morphological closing removes the ink and leaves the paper-plus-stain background. Otsu on that background marks where the paper is darkened by a stain.
2. **Masked Level 3.** Level 3 (p3, the eq. 8 / Fig. 5 reading) is applied inside the stain, and Level 2 everywhere else. Post-enhancement is then applied as in the paper.
3. **Reading output (not in the paper).** The ink keeps its grey tones instead of being binarised. Hairlines and pen pressure survive, which makes the result much easier to read by eye. Use the binary for OCR/HTR and the tone image for reading.

`close_size` must be wider than the thickest pen stroke. Raise it for higher-resolution scans: about 31 suits a ~1200 px wide image, and about 81 suits a 3000 px scan.

In [ ]:
def stain_mask(Gs, close_size=31):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_size, close_size))
    background = cv2.GaussianBlur(cv2.morphologyEx(Gs, cv2.MORPH_CLOSE, k), (0, 0), close_size / 2)
    t, _ = cv2.threshold(background, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return cv2.dilate((background < t).astype(np.uint8), k), background

def tone_output(g, T, ink_max=170):
    """Background white; ink grey levels [0, T) stretched to [0, ink_max]."""
    out = np.full_like(g, 255)
    ink = g < 255
    out[ink] = np.clip(g[ink].astype(np.float32) * ink_max / max(T, 1), 0, ink_max).astype(np.uint8)
    return out

def restore_zoernikav(bgr, close_size=31, dilate_size=2):
    r = tsat(bgr, level3_partition='p3')
    S, background = stain_mask(r['Gs'], close_size)
    hybrid = np.where(S == 1, r['g3'], r['g2'])
    r['stain_mask'] = S
    r['background'] = background
    r['binary_adapted'] = post_enhance(np.where(hybrid == 255, 255, 0).astype(np.uint8), dilate_size)
    # tone: Level 2 tones outside the stain, Level 3 tones inside, each scaled by its own threshold
    r['tone'] = np.where(S == 1, tone_output(r['g3'], r['T']['T3']), tone_output(r['g2'], r['T']['T2']))
    return r

z = restore_zoernikav(I)
overlay = I.copy()
overlay[z['stain_mask'] == 1] = (0.6 * overlay[z['stain_mask'] == 1] + 0.4 * np.array([0, 0, 255])).astype(np.uint8)
show([overlay, paper['binary'], z['binary_adapted'], z['tone']],
     ['stain mask (red)', 'paper method (Level 3, p2)', 'adapted: masked Level 3 — for OCR/HTR', 'adapted: tone output — for reading'])

### Close-up

Small, downscaled figures make any black-and-white image look broken, so judge the saved PNGs at 100% zoom.

In [ ]:
h, w = I.shape[:2]
y0, y1, x0, x1 = int(0.25 * h), int(0.65 * h), 0, int(0.6 * w)   # adjust to the stained area
crop = lambda a: a[y0:y1, x0:x1]
show([crop(I), crop(paper['binary']), crop(z['binary_adapted']), crop(z['tone'])],
     ['original', 'paper method', 'adapted binary', 'adapted tone'], cols=1, width=11)

## 5. Batch: every folio in `StainImages/`

In [ ]:
for p in image_paths:
    bgr = cv2.imread(p)
    z = restore_zoernikav(bgr)
    stem = os.path.splitext(os.path.basename(p))[0]
    cv2.imwrite(os.path.join(OUT_DIR, f'{stem}_paper.png'), z['binary'])
    cv2.imwrite(os.path.join(OUT_DIR, f'{stem}_binary.png'), z['binary_adapted'])
    cv2.imwrite(os.path.join(OUT_DIR, f'{stem}_tone.png'), z['tone'])
    T = z['T']
    print(f"{stem:30s} T1={T['T1']:6.1f}  T2={T['T2']:5.1f}  T3(p3)={T['T3']:5.1f}  "
          f"stain={100 * z['stain_mask'].mean():4.1f}%")
print(f'written to {OUT_DIR}/')

## 6. Baselines compared in the paper (§4, Fig. 12)

These are the classical thresholding methods the paper benchmarks against: Niblack, Otsu, Sauvola, Wolf, Kittler, Bradley, local Otsu and Bernsen. Window sizes and k values are the usual defaults; the paper does not state its own.

In [ ]:
def _local_stats(gray, w):
    f = gray.astype(np.float64)
    m = cv2.boxFilter(f, -1, (w, w))
    s = np.sqrt(np.clip(cv2.boxFilter(f * f, -1, (w, w)) - m * m, 0, None))
    return f, m, s

def niblack(gray, w=25, k=-0.2):
    f, m, s = _local_stats(gray, w)
    return np.where(f > m + k * s, 255, 0).astype(np.uint8)

def otsu(gray):
    return cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]

def sauvola(gray, w=25, k=0.2, R=128.0):
    f, m, s = _local_stats(gray, w)
    return np.where(f > m * (1 + k * (s / R - 1)), 255, 0).astype(np.uint8)

def wolf(gray, w=25, k=0.5):
    f, m, s = _local_stats(gray, w)
    M, Rm = f.min(), max(s.max(), 1e-6)
    return np.where(f > (1 - k) * m + k * M + k * s / Rm * (m - M), 255, 0).astype(np.uint8)

def kittler(gray):
    """Kittler-Illingworth minimum error thresholding."""
    h = np.bincount(gray.ravel(), minlength=256).astype(np.float64); h /= h.sum()
    g = np.arange(256); best, T = np.inf, 0
    for t in range(1, 255):
        P1, P2 = h[:t + 1].sum(), h[t + 1:].sum()
        if P1 < 1e-9 or P2 < 1e-9: continue
        m1 = (g[:t + 1] * h[:t + 1]).sum() / P1; m2 = (g[t + 1:] * h[t + 1:]).sum() / P2
        v1 = ((g[:t + 1] - m1) ** 2 * h[:t + 1]).sum() / P1; v2 = ((g[t + 1:] - m2) ** 2 * h[t + 1:]).sum() / P2
        if v1 <= 0 or v2 <= 0: continue
        J = 1 + 2 * (P1 * np.log(np.sqrt(v1)) + P2 * np.log(np.sqrt(v2))) - 2 * (P1 * np.log(P1) + P2 * np.log(P2))
        if J < best: best, T = J, t
    return np.where(gray > T, 255, 0).astype(np.uint8)

def bradley(gray, w=None, t=0.15):
    w = w or (max(gray.shape) // 8) | 1
    m = cv2.boxFilter(gray.astype(np.float64), -1, (w, w))
    return np.where(gray > m * (1 - t), 255, 0).astype(np.uint8)

def local_otsu(gray, tile=64):
    out = np.zeros_like(gray)
    for y in range(0, gray.shape[0], tile):
        for x in range(0, gray.shape[1], tile):
            out[y:y + tile, x:x + tile] = otsu(gray[y:y + tile, x:x + tile])
    return out

def bernsen(gray, w=31, contrast_min=15):
    k = np.ones((w, w), np.uint8)
    mx, mn = cv2.dilate(gray, k).astype(np.int32), cv2.erode(gray, k).astype(np.int32)
    mid = (mx + mn) / 2
    out = np.where(gray >= mid, 255, 0).astype(np.uint8)
    out[(mx - mn) < contrast_min] = np.where(mid[(mx - mn) < contrast_min] >= 128, 255, 0)
    return out

BASELINES = dict(Niblack=niblack, Otsu=otsu, Sauvola=sauvola, Wolf=wolf, Kittler=kittler,
                 Bradley=bradley, LocalOtsu=local_otsu, Bernsen=bernsen)

gray = cv2.cvtColor(I, cv2.COLOR_BGR2GRAY)
results = {'Proposed (paper)': paper['binary'], 'Adapted': z['binary_adapted']}
results.update({k: f(gray) for k, f in BASELINES.items()})
show(list(results.values()), list(results.keys()), cols=2, width=16)

## 7. Evaluation metrics (§4, Table 3)

These are the DIBCO measures the paper reports. Each takes the ground truth `gt` and the result `out` as images with ink = 0 and background = 255.

| metric | better when | definition |
|---|---|---|
| FM | ↑ | F-measure, 2PR/(P+R) × 100 |
| pFM | ↑ | pseudo-F-measure: recall measured on the skeleton of the ground truth |
| PSNR | ↑ | 10·log10(1/MSE) on the 0/1 images |
| NRM | ↓ | negative rate metric, (FN/(FN+TP) + FP/(FP+TN)) / 2 |
| DRD | ↓ | distance-reciprocal distortion (Lu et al. 2004), 5×5 weights, normalised by the number of non-uniform 8×8 blocks |
| MPM | ↓ | misclassification penalty metric: FN/FP pixels weighted by their distance to the ground-truth contour |

pFM here uses the plain skeleton recall of DIBCO 2010. The weighted pseudo-recall maps of DIBCO 2013+ are distributed with the dataset and are not reproduced here.

The stained Zoernikav folios have no ground truth. To score the method, point `GT_DIR` at a DIBCO set: the originals and `*_gt` / `*_GT` images in the same folder, or two parallel folders.

In [ ]:
from skimage.morphology import skeletonize

def _fg(img):                       # ink = True
    return img < 128

def evaluate(gt_img, out_img):
    gt, out = _fg(gt_img), _fg(out_img)
    TP = np.sum(gt & out); FP = np.sum(~gt & out); FN = np.sum(gt & ~out); TN = np.sum(~gt & ~out)
    P = TP / max(TP + FP, 1); R = TP / max(TP + FN, 1)
    FM = 100 * 2 * P * R / max(P + R, 1e-12)
    skel = skeletonize(gt)
    pR = np.sum(skel & out) / max(skel.sum(), 1)
    pFM = 100 * 2 * P * pR / max(P + pR, 1e-12)
    mse = (FP + FN) / gt.size
    PSNR = 10 * np.log10(1 / max(mse, 1e-12))
    NRM = (FN / max(FN + TP, 1) + FP / max(FP + TN, 1)) / 2

    # DRD
    W = np.array([[1 / np.hypot(i - 2, j - 2) if (i, j) != (2, 2) else 0 for j in range(5)] for i in range(5)])
    W /= W.sum()
    gtp = np.pad(gt.astype(np.float64), 2, mode='edge')
    ys, xs = np.nonzero(gt != out)
    drd_sum = 0.0
    for y, x in zip(ys, xs):
        drd_sum += np.sum(np.abs(gtp[y:y + 5, x:x + 5] - out[y, x]) * W)
    hh, ww = (gt.shape[0] // 8) * 8, (gt.shape[1] // 8) * 8
    blocks = gt[:hh, :ww].reshape(hh // 8, 8, ww // 8, 8).sum(axis=(1, 3))
    NUBN = np.sum((blocks > 0) & (blocks < 64))
    DRD = drd_sum / max(NUBN, 1)

    # MPM
    contour = gt & ~cv2.erode(gt.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
    dist = cv2.distanceTransform((~contour).astype(np.uint8), cv2.DIST_L2, 5)
    D = max(dist.sum(), 1e-12)
    MPM = (dist[gt & ~out].sum() / D + dist[~gt & out].sum() / D) / 2 * 1000   # ×1000 as DIBCO reports

    return dict(FM=FM, pFM=pFM, PSNR=PSNR, NRM=NRM, DRD=DRD, MPM=MPM)

# sanity check: a perfect result
print({k: round(v, 3) for k, v in evaluate(paper['binary'], paper['binary']).items()})

In [ ]:
GT_DIR = os.path.expanduser('~/Downloads/DIBC02009_Test_images-handwritten')   # change to your DIBCO folder

def find_pairs(folder):
    files = sorted(glob.glob(os.path.join(folder, '**', '*.*'), recursive=True))
    gts = {os.path.basename(f).lower().replace('_gt', '').rsplit('.', 1)[0]: f
           for f in files if '_gt' in os.path.basename(f).lower()}
    pairs = []
    for f in files:
        stem = os.path.basename(f).lower().rsplit('.', 1)[0]
        if '_gt' not in stem and stem in gts:
            pairs.append((f, gts[stem]))
    return pairs

pairs = find_pairs(GT_DIR) if os.path.isdir(GT_DIR) else []
print(f'{len(pairs)} image / ground-truth pairs in {GT_DIR}')

if pairs:
    import pandas as pd
    rows = []
    for img_p, gt_p in pairs:
        img = cv2.imread(img_p); gt = cv2.imread(gt_p, cv2.IMREAD_GRAYSCALE)
        if img.ndim == 2 or img.shape[2] == 1: img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        methods = {'Proposed': tsat(img)['binary'], **{k: f(g) for k, f in BASELINES.items()}}
        for name, out in methods.items():
            rows.append(dict(image=os.path.basename(img_p), method=name, **evaluate(gt, out)))
    df = pd.DataFrame(rows)
    display(df.groupby('method')[['FM', 'pFM', 'PSNR', 'NRM', 'DRD', 'MPM']].mean().round(2).sort_values('FM', ascending=False))